In [16]:
# %%
# !pip install youtube-transcript-api scrapetube

from youtube_transcript_api import YouTubeTranscriptApi
import scrapetube
from urllib.parse import urlparse, parse_qs
import os

In [17]:
# %%
# --- Pipeline Configuration ---
# Feed the queue channel handles or URLs instead of hardcoded video links.

experts_queue = {
    "josh_braun": "https://www.youtube.com/@joshbraunsales",
    "guillaume_moubeche": "https://www.youtube.com/@GuillaumeMoubeche",
    "jason_bay": "https://www.youtube.com/@jasondbay", 
    "alex_berman": "https://www.youtube.com/@AlexBerman",
    "jeremy_chatelaine": "https://www.youtube.com/@outreachwithjeremy",
    "john_barrows": "https://www.youtube.com/@JohnMBarrows",
    "nick_abraham": "https://www.youtube.com/@nickabraham12",
    "becc_holland": "https://www.youtube.com/@flipthescript-sales",
    "will_allred": "https://www.youtube.com/@itslavenderduh",
    "jed_mahrle": "https://www.youtube.com/@practicalprospecting"
}

# Number of recent videos to pull per expert
MAX_VIDEOS_PER_EXPERT = 2

OUTPUT_DIR = "../research/youtube-transcripts"

In [18]:
# %%
def extract_video_id(url: str) -> str:
    """Robust URL parser for all YouTube link formats."""
    parsed = urlparse(url)
    if parsed.hostname == 'youtu.be':
        return parsed.path[1:]
    if parsed.hostname in ('www.youtube.com', 'youtube.com'):
        if parsed.path == '/watch':
            return parse_qs(parsed.query).get('v', [None])[0]
    return url

In [19]:
# %%
# Robust Extraction Layer - Checks all content types (Videos, Shorts, Streams)
transcripts_data = []
ytt_api = YouTubeTranscriptApi()

# We'll check these tabs in order of "signal quality"
CONTENT_TYPES = ["videos", "streams", "shorts"]

for expert, channel_url in experts_queue.items():
    if not channel_url: continue
        
    print(f"\nScanning channel for {expert}...")
    found_videos = []
    
    # Try each content type until we hit our MAX_VIDEOS_PER_EXPERT limit
    for c_type in CONTENT_TYPES:
        if len(found_videos) >= MAX_VIDEOS_PER_EXPERT:
            break
            
        try:
            limit_needed = MAX_VIDEOS_PER_EXPERT - len(found_videos)
            vids = scrapetube.get_channel(channel_url=channel_url, 
                                         limit=limit_needed, 
                                         content_type=c_type)
            found_videos.extend(list(vids))
        except Exception as e:
            continue # If a tab doesn't exist, just move to the next

    print(f"  Found {len(found_videos)} recent items across tabs. Extracting...")
    
    for vid in found_videos:
        vid_id = vid['videoId']
        try:
            raw_transcript = ytt_api.fetch(vid_id)
            full_text = " ".join([segment.text for segment in raw_transcript])
            
            transcripts_data.append({
                "expert": expert,
                "video_id": vid_id,
                "url": f"https://www.youtube.com/watch?v={vid_id}",
                "content": full_text.replace(". ", ".\n\n")
            })
            print(f"    ✓ Success: [{vid_id}]")
        except Exception as e:
            print(f"    ✗ Transcript not available for [{vid_id}]: {e}")

print("\nRobust Extraction Complete.")


Scanning channel for josh_braun...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [hpXev0vUcvQ]
    ✓ Success: [_gnog7udq7A]

Scanning channel for guillaume_moubeche...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [swSBRiY9C_g]
    ✓ Success: [WqZzC8Wragw]

Scanning channel for jason_bay...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [qc2IY_pNXhE]
    ✓ Success: [WvVNB-2ylSg]

Scanning channel for alex_berman...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [kCIz_-JSAJw]
    ✓ Success: [Ikndrts8NQU]

Scanning channel for jeremy_chatelaine...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [wFzoXKha4gQ]
    ✓ Success: [ykzmySZtNcc]

Scanning channel for john_barrows...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [w9T7LFW6yV0]
    ✓ Success: [yqlFUHLbPxs]

Scanning channel for nick_abraham...
  Found 2 recent items across tabs. Extracting...
    ✓ Success: [SnPRR1qMhYA]
  

In [20]:
# %%
import glob

# Ensure the target directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# THE UPGRADE: Clear out old markdown files before saving the new batch
print("Sweeping the directory for old data...")
old_files = glob.glob(os.path.join(OUTPUT_DIR, "*.md"))
for f in old_files:
    os.remove(f)

for data in transcripts_data:
    # Flat file naming convention: expert_videoid.md
    filename = os.path.join(OUTPUT_DIR, f"{data['expert']}_{data['video_id']}.md")
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Transcript: {data['expert'].replace('_', ' ').title()}\n\n")
        f.write(f"**Video ID:** {data['video_id']}\n")
        f.write(f"**Source URL:** {data['url']}\n\n")
        f.write("---\n\n")
        f.write("### Content\n\n")
        f.write(data['content'])

print("Files saved to storage. Performing directory audit...\n")

# QA check to verify file sizes
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"  {file} — ({size} bytes)")

Sweeping the directory for old data...
Files saved to storage. Performing directory audit...

  alex_berman_Ikndrts8NQU.md — (5876 bytes)
  alex_berman_kCIz_-JSAJw.md — (5297 bytes)
  becc_holland_Qa-3XhyNOIM.md — (87591 bytes)
  becc_holland_y6CO23qfRDM.md — (40470 bytes)
  guillaume_moubeche_swSBRiY9C_g.md — (69163 bytes)
  guillaume_moubeche_WqZzC8Wragw.md — (59338 bytes)
  jason_bay_qc2IY_pNXhE.md — (65386 bytes)
  jason_bay_WvVNB-2ylSg.md — (937 bytes)
  jed_mahrle_9BIrXMfOMHw.md — (21751 bytes)
  jed_mahrle_dUa-09OF_BI.md — (22747 bytes)
  jeremy_chatelaine_wFzoXKha4gQ.md — (12499 bytes)
  jeremy_chatelaine_ykzmySZtNcc.md — (14558 bytes)
  john_barrows_w9T7LFW6yV0.md — (2779 bytes)
  john_barrows_yqlFUHLbPxs.md — (4358 bytes)
  josh_braun_hpXev0vUcvQ.md — (2167 bytes)
  josh_braun__gnog7udq7A.md — (2375 bytes)
  nick_abraham_SnPRR1qMhYA.md — (1743 bytes)
  nick_abraham_ZkFjRolCiHA.md — (1202 bytes)
  will_allred_b0suhvh-s0I.md — (1558 bytes)
  will_allred_QLlLDJvaaJA.md — (2271 b